# Notebook 3: LSTM Autoencoder — Anomaly Detection
## ESP Predictive Maintenance

**Approach**: Train a Bidirectional LSTM Autoencoder on **normal** sensor windows only.
After training, the model reconstructs normal patterns well but fails to reconstruct
anomalous (degraded/failure) patterns → high reconstruction error = anomaly alarm.

**Monte Carlo Dropout** provides uncertainty quantification:
run N stochastic forward passes → mean anomaly score ± confidence interval.

**Colab instructions**:
- Runtime → Change runtime type → GPU (T4 is fine for this model)
- If saving to Drive, ensure it is mounted in Cell 1


In [ ]:
# ── Cell 1: Environment setup ─────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q torch torchvision tqdm scikit-learn imbalanced-learn shap
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/esp_pm/checkpoints/lstm_ae'
else:
    SAVE_DIR = '../checkpoints/lstm_ae'

import os
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs('../results', exist_ok=True)

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Project imports
sys.path.insert(0, os.path.abspath('..'))
from src.data.loader import make_dataloaders, TimeSeriesDataset
from src.data.synthetic_generator import generate_esp_dataset, SYNTHETIC_SENSOR_COLS
from src.models.lstm_autoencoder import LSTMAutoencoder, mc_dropout_anomaly_scores
from src.training.trainer import AutoencoderTrainer
from src.utils.metrics import anomaly_detection_metrics, early_detection_lead_time
from src.utils.visualization import plot_anomaly_scores, plot_reconstruction_comparison

plt.rcParams.update({'figure.facecolor': 'white', 'axes.grid': True, 'grid.alpha': 0.3})
print('Imports OK.')

In [ ]:
# ── Cell 3: Hyperparameters ───────────────────────────────────────
# ✏️  Tune these!
CONFIG = {
    # Data
    'window_size':    50,     # timesteps per input window
    'step_size':       1,     # sliding window step
    'val_split':    0.15,
    'test_split':   0.15,

    # Model
    'hidden_size':   128,
    'num_layers':      2,
    'latent_size':    32,
    'dropout':       0.3,

    # Training
    'batch_size':    128,
    'num_epochs':    100,
    'lr':           1e-3,
    'weight_decay': 1e-4,
    'patience':       15,
    'gradient_clip': 1.0,

    # Anomaly detection
    'threshold_pct': 95,      # calibration percentile on normal windows
    'mc_samples':     50,     # MC Dropout forward passes
}

print('CONFIG:')
for k, v in CONFIG.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# ── Cell 4: Load & prepare data ──────────────────────────────────
# ✏️  CHOOSE YOUR DATA SOURCE
# Option A: Synthetic (generated on the fly)
# Option B: Real Pump Sensor CSV from Google Drive

USE_REAL_DATA = False   # ← Set True to load from Drive

if USE_REAL_DATA:
    # ── Real Pump Sensor Data from Drive ──────────────────────
    # Expected CSV: sensor.csv from Kaggle pump-sensor-data
    # Has columns: timestamp, sensor_00, ..., sensor_51, machine_status
    DRIVE_CSV_PATH = '/content/drive/MyDrive/pump_sensor.csv'  # ← change if needed
    print(f'Loading real data from {DRIVE_CSV_PATH}...')

    raw_df = pd.read_csv(DRIVE_CSV_PATH, parse_dates=['timestamp'])
    raw_df = raw_df.sort_values('timestamp').reset_index(drop=True)

    # Detect sensor columns (sensor_XX pattern)
    SENSOR_COLS = sorted([c for c in raw_df.columns if c.startswith('sensor_')])
    print(f'Found {len(SENSOR_COLS)} sensor columns')

    # Forward-fill missing values, then fill remaining with 0
    raw_df[SENSOR_COLS] = raw_df[SENSOR_COLS].ffill().bfill()

    # Binary failure label
    status_map = {'NORMAL': 0, 'RECOVERING': 0, 'BROKEN': 1}
    raw_df['failure'] = raw_df['machine_status'].map(status_map).fillna(0).astype(int)
    print(f'Raw data: {len(raw_df):,} rows, {len(SENSOR_COLS)} sensors')
    print(f'Failure rate: {raw_df["failure"].mean()*100:.2f}%')

else:
    # ── Synthetic Data ────────────────────────────────────────
    from src.data.synthetic_generator import generate_esp_dataset, SYNTHETIC_SENSOR_COLS
    print('Generating synthetic ESP data...')
    raw_df = generate_esp_dataset(n_wells=40, timesteps_per_well=4000, random_seed=42)
    SENSOR_COLS = SYNTHETIC_SENSOR_COLS
    raw_df['failure'] = (raw_df['machine_status'] == 'BROKEN').astype(int)
    print(f'Raw data: {len(raw_df):,} rows, {len(SENSOR_COLS)} sensors')

# Build sliding windows
from src.data.loader import _sliding_window, _compute_rul, _split_and_scale

X_raw = raw_df[SENSOR_COLS].values.astype(np.float32)
y_raw = raw_df['failure'].values.astype(np.float32)
rul_raw = _compute_rul(y_raw)

X_w, y_w, rul_w = _sliding_window(
    X_raw, y_raw, rul_raw,
    CONFIG['window_size'], CONFIG['step_size']
)
print(f'Windows: {X_w.shape}, failures: {y_w.sum():.0f} ({y_w.mean()*100:.2f}%)')

data = _split_and_scale(
    X_w, y_w, rul_w, SENSOR_COLS,
    val_split=CONFIG['val_split'],
    test_split=CONFIG['test_split'],
    random_seed=42
)

In [ ]:
# ── Cell 5: Create DataLoaders ────────────────────────────────────
# KEY: Train autoencoder on NORMAL windows only
normal_train_mask = data['y_train'] == 0
X_train_normal = data['X_train'][normal_train_mask]
print(f'Normal training windows: {X_train_normal.shape[0]:,}')

train_ds_normal = TimeSeriesDataset(X_train_normal)
val_ds          = TimeSeriesDataset(data['X_val'], data['y_val'])
test_ds         = TimeSeriesDataset(data['X_test'], data['y_test'])

num_workers = 0  # 0 = safest on Colab
train_loader = DataLoader(train_ds_normal, batch_size=CONFIG['batch_size'],
                          shuffle=True, num_workers=num_workers)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=num_workers)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=num_workers)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# ── Cell 6: Build model ───────────────────────────────────────────
n_features = data['X_train'].shape[2]  # number of sensor channels

model = LSTMAutoencoder(
    input_size=n_features,
    hidden_size=CONFIG['hidden_size'],
    num_layers=CONFIG['num_layers'],
    latent_size=CONFIG['latent_size'],
    dropout=CONFIG['dropout'],
    seq_len=CONFIG['window_size'],
    bidirectional_encoder=True,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: LSTMAutoencoder')
print(f'Parameters: {n_params:,}')
print(f'Input: ({CONFIG["window_size"]}, {n_features}) → Latent: ({CONFIG["latent_size"]},)')
print(model)

In [ ]:
# ── Cell 7: Train ────────────────────────────────────────────────
trainer = AutoencoderTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    save_dir=SAVE_DIR,
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
    gradient_clip=CONFIG['gradient_clip'],
    early_stopping_patience=CONFIG['patience'],
    teacher_forcing_ratio=0.5,
    teacher_forcing_decay=0.005,
)

print('Training LSTM Autoencoder...')
history = trainer.train(num_epochs=CONFIG['num_epochs'])

fig = trainer.plot_history(save_path='../results/03_training_curves.png')
plt.show()
print(f'Best val loss: {min(history["val_loss"]):.6f}')

In [ ]:
# ── Cell 8: Calibrate anomaly threshold ──────────────────────────
# Threshold = 95th percentile of reconstruction error on NORMAL windows
normal_test_mask = data['y_test'] == 0
X_test_normal = data['X_test'][normal_test_mask]
normal_calibrate_ds = TimeSeriesDataset(X_test_normal)
normal_calibrate_loader = DataLoader(normal_calibrate_ds, batch_size=256, shuffle=False)

threshold = model.calibrate_threshold(
    normal_calibrate_loader,
    device=DEVICE,
    percentile=CONFIG['threshold_pct'],
)
print(f'Anomaly threshold (P{CONFIG["threshold_pct"]}): {threshold:.6f}')
print('Interpretation: reconstruction errors above this value → anomaly alarm')

In [ ]:
# ── Cell 9: Evaluation ────────────────────────────────────────────
# Compute anomaly scores on full test set
all_scores = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        x = batch['X'].to(DEVICE)
        scores = model.anomaly_score(x)
        all_scores.append(scores.cpu().numpy())

test_scores = np.concatenate(all_scores)
y_test = data['y_test']

metrics = anomaly_detection_metrics(y_test, test_scores, threshold=threshold)

print('='*55)
print('LSTM AUTOENCODER — TEST SET RESULTS')
print('='*55)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:25s}: {v:.4f}')
    else:
        print(f'  {k:25s}: {v}')
print('='*55)

In [ ]:
# ── Cell 10: MC Dropout — uncertainty quantification ──────────────
print(f'Running {CONFIG["mc_samples"]} MC Dropout forward passes...')

# Use a subset for speed
X_sample = torch.from_numpy(data['X_test'][:2000]).float().to(DEVICE)
mc_mean, mc_std, _ = mc_dropout_anomaly_scores(
    model, X_sample, n_samples=CONFIG['mc_samples']
)
y_sample = y_test[:2000]

fig = plot_anomaly_scores(
    timestamps=np.arange(len(mc_mean)),
    scores=mc_mean,
    y_true=y_sample,
    threshold=threshold,
    mc_std=mc_std,
    title='LSTM Autoencoder — Anomaly Score with MC Dropout Uncertainty',
    save_path='../results/03_anomaly_scores_mc.png',
)
plt.show()
print(f'Mean uncertainty (std): {mc_std.mean():.6f}')
print('Tip: High uncertainty near threshold = borderline windows → flag for human review')

In [ ]:
# ── Cell 11: Reconstruction visualisation ─────────────────────────
# Find a failure window to visualise
failure_idxs = np.where(y_test == 1)[0]
if len(failure_idxs) > 0:
    sample_idx = failure_idxs[0]
    x_sample = torch.from_numpy(data['X_test'][sample_idx:sample_idx+1]).float().to(DEVICE)
    model.eval()
    with torch.no_grad():
        x_hat, _ = model(x_sample, teacher_forcing_ratio=0.0)

    fig = plot_reconstruction_comparison(
        x_original=data['X_test'][sample_idx:sample_idx+1],
        x_reconstructed=x_hat.cpu().numpy(),
        sensor_names=SENSOR_COLS[:6],
        sample_idx=0,
        n_sensors_to_show=6,
        title='LSTM AE — Reconstruction of a Failure Window',
        save_path='../results/03_reconstruction_failure.png',
    )
    plt.show()
    score = test_scores[sample_idx]
    print(f'Reconstruction error for this failure window: {score:.6f}')
    print(f'Threshold: {threshold:.6f}  →  Anomaly: {score > threshold}')

In [ ]:
# ── Cell 12: Save model ───────────────────────────────────────────
model.save_pretrained(SAVE_DIR)
print(f'Model saved to {SAVE_DIR}')
print(f'Files: {os.listdir(SAVE_DIR)}')

# Also save config for HuggingFace upload
import json
full_config = {**CONFIG, **model.get_config(), 'sensor_cols': SENSOR_COLS}
with open(os.path.join(SAVE_DIR, 'training_config.json'), 'w') as f:
    json.dump(full_config, f, indent=2)

print('\nNext steps:')
print('  1. Open notebooks/04_Transformer_Anomaly_Detection.ipynb')
print('  2. Open notebooks/05_RUL_Prediction.ipynb')
print('  3. When all models trained: python scripts/upload_to_hf.py --model lstm ...')